In [ ]:
!apt-get update -qq && apt-get install -y -qq \
  build-essential python3-dev pkg-config \
  libosmesa6-dev patchelf libglew-dev libglfw3

In [ ]:
%%bash
set -e
cd /content
if [ ! -d PIToD/.git ]; then
  git clone https://github.com/NehaBhat14/PIToD.git PIToD
fi
cd PIToD
git pull


In [ ]:
# Mount Google Drive and symlink runs/ + figure/ so artifacts persist.
from google.colab import drive
import os
import subprocess

drive.mount('/content/drive', force_remount=False)
repo = '/content/PIToD'
persist_root = '/content/drive/MyDrive/PIToD_runs'
for name in ('runs', 'figure', 'logs'):
    os.makedirs(os.path.join(persist_root, name), exist_ok=True)
for name in ('runs', 'figure'):
    target = os.path.join(persist_root, name)
    link = os.path.join(repo, name)
    if os.path.islink(link):
        continue
    if os.path.isdir(link):
        subprocess.run(['rm', '-rf', link], check=True)
    os.symlink(target, link)
subprocess.run(['ls', '-la', os.path.join(repo, 'runs'), os.path.join(repo, 'figure')], check=False)


In [ ]:
%%bash
set -euo pipefail
MJ="${HOME}/.mujoco"
ARCH="${MJ}/mujoco210-linux-x86_64.tar.gz"
mkdir -p "${MJ}"
if [ ! -f "${MJ}/mujoco210/bin/libmujoco210.so" ]; then
  wget -nv -O "${ARCH}" \
    "https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz" \
    || curl -fL -o "${ARCH}" \
    "https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz"
  tar -xzf "${ARCH}" -C "${MJ}"
fi
test -f "${MJ}/mujoco210/bin/libmujoco210.so"
echo "MuJoCo OK at ${MJ}/mujoco210"


In [ ]:
%%bash
set -e
export MUJOCO_PY_MUJOCO_PATH="${HOME}/.mujoco/mujoco210"
export LD_LIBRARY_PATH="${MUJOCO_PY_MUJOCO_PATH}/bin:${LD_LIBRARY_PATH:-}"
pip install -r /content/PIToD/requirements-colab.txt


In [ ]:
%%bash
set -e
export MUJOCO_PY_MUJOCO_PATH="${HOME}/.mujoco/mujoco210"
export LD_LIBRARY_PATH="${MUJOCO_PY_MUJOCO_PATH}/bin:${LD_LIBRARY_PATH:-}"
pip install "mujoco-py==2.1.2.14" -v


In [ ]:
# Focused static_pitod vs dynamic_pitod screen (see HOW_TO_RUN.md §8).
INFO = 'screen_dyn_early_vs_static'
ENV = 'Hopper-v2'
SEEDS = [0, 1, 2]
EPOCHS = 60
STEPS_PER_EPOCH = 5000
GPU_ID = 0
ANALYSIS_SEED = SEEDS[0]
RETURN_THRESHOLD = 1000.0

BASE_FLAGS = [
    '-env', ENV,
    '-epochs', str(EPOCHS), '-steps_per_epoch', str(STEPS_PER_EPOCH),
    '-info', INFO, '-gpu_id', str(GPU_ID),
    '-layer_norm', '1', '-layer_norm_policy', '1',
    '-evaluate_bias', '1', '--experience_cleansing', '1',
    '-n_eval', '20', '--influence_estimation_interval', '10', '--return_evaluation_interval', '5',
]

import os
import subprocess
import sys


def run_streaming(argv, log_name):
    env = os.environ.copy()
    env.update({
        'PYTHONPATH': '/content/PIToD',
        'PYTHONUNBUFFERED': '1',
        'MUJOCO_PY_MUJOCO_PATH': f"{os.environ['HOME']}/.mujoco/mujoco210",
        'LD_LIBRARY_PATH': f"{os.environ['HOME']}/.mujoco/mujoco210/bin:" + env.get('LD_LIBRARY_PATH', ''),
        'MUJOCO_GL': 'osmesa',
    })
    log_path = f"/content/drive/MyDrive/PIToD_runs/logs/{log_name}.log"
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'w', encoding='utf-8', errors='replace') as logf:
        p = subprocess.Popen(
            argv,
            cwd='/content/PIToD',
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            text=True,
        )
        assert p.stdout is not None
        for line in p.stdout:
            print(line, end='', flush=True)
            logf.write(line)
            logf.flush()
        p.wait()
    print(f"\n[exit {p.returncode}] log -> {log_path}")
    return p.returncode


In [ ]:
# static_pitod = uniform replay + post-hoc PIToD logging.
for seed in SEEDS:
    rc = run_streaming(
        [
            sys.executable,
            '-u',
            'dynamic-main-TH.py',
            *BASE_FLAGS,
            '-seed',
            str(seed),
            '--replay_mode',
            'static_pitod',
        ],
        log_name=f'{INFO}_static_pitod_s{seed}',
    )
    assert rc == 0, f'static_pitod failed for seed {seed} with exit {rc}'


In [ ]:
# Dynamic PIToD with earlier refresh and delayed pruning.
for seed in SEEDS:
    rc = run_streaming(
        [
            sys.executable,
            '-u',
            'dynamic-main-TH.py',
            *BASE_FLAGS,
            '-seed',
            str(seed),
            '--replay_mode',
            'dynamic_pitod',
            '--k_refresh',
            '10000',
            '--b_refresh',
            '16',
            '--dynamic_warmup_steps',
            '5000',
            '--early_phase_steps',
            '50000',
            '--early_k_refresh',
            '5000',
            '--early_b_refresh',
            '16',
            '--m_strikes',
            '5',
            '--prune_warmup_steps',
            '50000',
            '--pitod_alpha',
            '0.6',
            '--n_samples_per_group',
            '64',
            '--h2_log',
            '1',
        ],
        log_name=f'{INFO}_dynamic_pitod_s{seed}',
    )
    assert rc == 0, f'dynamic_pitod failed for seed {seed} with exit {rc}'


In [ ]:
import bz2
import glob
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = f'/content/PIToD/runs/{INFO}'
methods = ('static_pitod', 'dynamic_pitod')
dynamic_cols = [
    'AverageDynPIToD/ScoreMean',
    'AverageDynPIToD/Epsilon',
    'AverageDynPIToD/NumActive',
    'AverageDynPIToD/BufferActiveFrac',
    'AverageDynPIToD/RefreshWallclock',
    'AverageDynPIToD/NumRefreshed',
]

def trapz_auc(x, y):
    if hasattr(np, 'trapezoid'):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))

def first_crossing(xs, ys, threshold):
    crossed = np.flatnonzero(np.asarray(ys) >= threshold)
    if crossed.size == 0:
        return float('nan')
    return float(np.asarray(xs)[crossed[0]])

runs_by_method = {m: {} for m in methods}
per_seed_rows = []
curve_rows = []

for m in methods:
    for seed in SEEDS:
        pat = f'{root}/redq_sac_{ENV}_{m}/redq_sac_{ENV}_{m}_s{seed}'
        hits = sorted(glob.glob(pat))
        if not hits:
            raise FileNotFoundError(f'Missing run directory: {pat}')
        run_dir = hits[0]
        runs_by_method[m][seed] = run_dir

        df_full = pd.read_table(os.path.join(run_dir, 'progress.txt'))
        keep = ['Epoch', 'AverageTestEpRet', 'AverageQBias', 'Time']
        keep += [c for c in dynamic_cols if c in df_full.columns]
        df = df_full[keep].copy()
        df['seed'] = seed
        df['method'] = m
        curve_rows.append(df)

        per_seed_rows.append({
            'method': m,
            'seed': seed,
            'final': float(df['AverageTestEpRet'].iloc[-1]),
            'best': float(df['AverageTestEpRet'].max()),
            'auc': trapz_auc(df['Epoch'].to_numpy(), df['AverageTestEpRet'].to_numpy()),
            'wallclock_s': float(df['Time'].iloc[-1]),
            'threshold_epoch': first_crossing(df['Epoch'], df['AverageTestEpRet'], RETURN_THRESHOLD),
            'threshold_time_s': first_crossing(df['Time'], df['AverageTestEpRet'], RETURN_THRESHOLD),
        })

curves = pd.concat(curve_rows, ignore_index=True)
summary_df = pd.DataFrame(per_seed_rows).sort_values(['method', 'seed'])
agg_df = (
    summary_df.groupby('method', as_index=False)
    .agg(
        final_mean=('final', 'mean'),
        final_std=('final', 'std'),
        best_mean=('best', 'mean'),
        best_std=('best', 'std'),
        auc_mean=('auc', 'mean'),
        auc_std=('auc', 'std'),
        wallclock_mean=('wallclock_s', 'mean'),
        wallclock_std=('wallclock_s', 'std'),
        threshold_epoch_mean=('threshold_epoch', 'mean'),
        threshold_epoch_std=('threshold_epoch', 'std'),
        threshold_time_mean=('threshold_time_s', 'mean'),
        threshold_time_std=('threshold_time_s', 'std'),
    )
)

print('\n== Per-seed summary (higher AUC/return = better for learning) ==')
for row in summary_df.itertuples(index=False):
    crossing = row.threshold_epoch if np.isfinite(row.threshold_epoch) else 'never'
    print(
        f"{row.method:15s} seed={row.seed} final={row.final:7.1f} "
        f"best={row.best:7.1f} auc={row.auc:10.1f} wallclock={row.wallclock_s:.0f}s "
        f"cross@{RETURN_THRESHOLD:.0f}={crossing}"
    )

print('\n== Aggregate summary (mean +/- std across seeds) ==')
for row in agg_df.itertuples(index=False):
    crossing = row.threshold_epoch_mean if np.isfinite(row.threshold_epoch_mean) else 'never'
    print(
        f"{row.method:15s} final={row.final_mean:7.1f} +/- {row.final_std:6.1f} "
        f"best={row.best_mean:7.1f} +/- {row.best_std:6.1f} "
        f"auc={row.auc_mean:10.1f} +/- {row.auc_std:7.1f} "
        f"wallclock={row.wallclock_mean:.0f}s +/- {row.wallclock_std:.0f}s "
        f"cross@{RETURN_THRESHOLD:.0f}={crossing}"
    )

mean_curve = (
    curves.groupby(['method', 'Epoch'], as_index=False)
    .agg(
        ret_mean=('AverageTestEpRet', 'mean'),
        ret_std=('AverageTestEpRet', 'std'),
        qbias_mean=('AverageQBias', 'mean'),
        qbias_std=('AverageQBias', 'std'),
    )
)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for m in methods:
    d = mean_curve[mean_curve['method'] == m].sort_values('Epoch')
    x = d['Epoch'].to_numpy()
    ret_mean = d['ret_mean'].to_numpy()
    ret_std = np.nan_to_num(d['ret_std'].to_numpy(), nan=0.0)
    ax[0].plot(x, ret_mean, label=m)
    ax[0].fill_between(x, ret_mean - ret_std, ret_mean + ret_std, alpha=0.2)
    q_mean = d['qbias_mean'].to_numpy()
    q_std = np.nan_to_num(d['qbias_std'].to_numpy(), nan=0.0)
    ax[1].plot(x, q_mean, label=m)
    ax[1].fill_between(x, q_mean - q_std, q_mean + q_std, alpha=0.2)

ax[0].set_title('AverageTestEpRet (mean +/- std over seeds)')
ax[0].set_xlabel('epoch')
ax[0].set_ylabel('return')
ax[0].legend()
ax[1].set_title('AverageQBias (mean +/- std over seeds)')
ax[1].set_xlabel('epoch')
ax[1].set_ylabel('q-bias')
ax[1].legend()
plt.tight_layout()
fig.savefig('/content/PIToD/figure/compare_learning_multiseed.png', dpi=120)
plt.show()

available_dynamic = [c for c in dynamic_cols if c in curves.columns]
if available_dynamic:
    fig_dyn, ax_dyn = plt.subplots(1, len(available_dynamic), figsize=(4 * len(available_dynamic), 4))
    if len(available_dynamic) == 1:
        ax_dyn = [ax_dyn]
    dyn_curves = curves[curves['method'] == 'dynamic_pitod'].copy()
    dyn_mean = dyn_curves.groupby('Epoch', as_index=False).mean(numeric_only=True)
    for axis, col in zip(ax_dyn, available_dynamic):
        axis.plot(dyn_mean['Epoch'], dyn_mean[col])
        axis.set_title(col.split('/')[-1])
        axis.set_xlabel('epoch')
    plt.tight_layout()
    fig_dyn.savefig('/content/PIToD/figure/dynamic_diagnostics.png', dpi=120)
    plt.show()

fig2, ax2 = plt.subplots(1, 3, figsize=(15, 4))
cleansing_files = [
    'list_q_bias_cleansing.bz2',
    'list_q_bias_cleansing_valid.bz2',
    'list_return_cleansing.bz2',
]
interval = 5
print(f'\nDetailed cleansing plots use ANALYSIS_SEED={ANALYSIS_SEED}')
for m in methods:
    run_dir = runs_by_method[m].get(ANALYSIS_SEED)
    if run_dir is None:
        continue
    for j, fname in enumerate(cleansing_files):
        path = os.path.join(run_dir, fname)
        if not os.path.isfile(path):
            continue
        with bz2.BZ2File(path, 'rb') as fh:
            arr = np.array(pickle.load(fh))[:, :, 0]
        x = np.arange(arr.shape[0]) * interval
        ax2[j].plot(x, arr[:, 0], '--', label=f'{m} pre')
        ax2[j].plot(x, arr[:, 1], '-', label=f'{m} post')
        ax2[j].set_title(fname.replace('.bz2', ''))
        ax2[j].set_xlabel('epoch')
ax2[0].legend()
plt.tight_layout()
fig2.savefig('/content/PIToD/figure/compare_cleansing_seed.png', dpi=120)
plt.show()
